In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Inițializarea Qubit-ului

<details>
<summary><b>Versiuni de pachete</b></summary>

Codul de pe această pagină a fost dezvoltat folosind următoarele cerințe.
Recomandăm utilizarea acestor versiuni sau a unora mai noi.

```
qiskit-ibm-runtime~=0.43.1
```
</details>
Când un Circuit este executat pe o unitate de procesare cuantică (QPU) IBM&reg;, la începutul circuitului este inserat de obicei un reset implicit pentru a asigura că Qubit-urile sunt inițializate la zero. Aceasta este controlată de indicatorul `init_qubits`, setat ca [opțiune de execuție a primitivelor](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-execution-options-v2).

Cu toate acestea, procesul de reset nu este perfect, ceea ce duce la erori de pregătire a stării. Pentru a reduce eroarea, sistemul inserează, de asemenea, un timp de întârziere de repetiție (sau `rep_delay`) între circuite. Fiecare Backend are un `rep_delay` implicit diferit, dar de obicei este mai lung decât T1 pentru a permite mediului să reseteze Qubit-urile. `rep_delay`-ul implicit poate fi interogat executând `backend.default_rep_delay`.

Toate QPU-urile IBM folosesc execuția cu rată de repetiție dinamică, ceea ce îți permite să modifici `rep_delay` pentru fiecare job. Circuitele pe care le trimiți într-un job primitiv sunt grupate împreună pentru execuție pe QPU. Aceste circuite sunt executate prin iterarea peste circuite pentru fiecare shot solicitat; execuția este pe coloane într-o matrice de circuite și shots, după cum este ilustrat în figura următoare.

![Prima coloană reprezintă shot-ul 0. Circuitele sunt rulate în ordine de la 0 la 3. A doua coloană reprezintă shot-ul 1. Circuitele sunt rulate în ordine de la 0 la 3. Coloanele rămase urmează același tipar. ](../docs/images/guides/repetition-rate-execution/circuits_shots_matrix1.avif "Matricea de execuție pe coloane")

Deoarece `rep_delay` este inserat între circuite, fiecare shot al execuției întâlnește această întârziere. Prin urmare, scăderea `rep_delay`-ului reduce timpul total de execuție pe QPU, dar cu prețul creșterii ratei de eroare de pregătire a stării, după cum se poate observa în imaginea următoare:

![Această imagine arată că, pe măsură ce valoarea `rep_delay` scade, rata de eroare de pregătire a stării crește.](../docs/images/guides/repetition-rate-execution/rep_delay.avif "Întârzierea de repetiție versus rata de eroare")

Setarea atât a `rep_delay=0`, cât și a `init_qubits=False` în esență „îmbină" circuitele, deoarece Qubit-urile vor începe în starea finală din shot-ul anterior.

Reține că, deși circuitele dintr-un job primitiv sunt grupate împreună pentru execuție pe QPU, nu există nicio garanție privind ordinea în care sunt executate circuitele din PUB-uri. Astfel, chiar dacă trimiți `pubs=[pub1, pub2]`, nu există nicio garanție că circuitele din `pub1` vor rula înaintea celor din `pub2`. De asemenea, nu există nicio garanție că circuitele din același job vor rula ca un singur lot pe QPU.

## Specificarea rep_delay pentru un job primitiv

In [1]:
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler

service = QiskitRuntimeService()

# Make sure your backend supports it
backend = service.least_busy(
    operational=True, min_num_qubits=100, dynamic_reprate_enabled=True
)

# Determine the allowable range
backend.rep_delay_range
sampler = Sampler(mode=backend)

# Specify a value in the supported range
sampler.options.execution.rep_delay = 0.0005

## Pași următori
> **Tip:** - Încearcă un exemplu în tutorialul [Algoritmul cuantic de optimizare aproximativă (QAOA)](/tutorials/quantum-approximate-optimization-algorithm).
>   - Analizează cum să [începi cu primitivele.](./get-started-with-primitives)